In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON, XML
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
from rdflib import Graph, URIRef, RDFS
import csv
import re
from collections import Counter, defaultdict
import csv
import requests
import time
import os
import urllib.parse
from bs4 import BeautifulSoup
from urllib.parse import unquote
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Optional, Sequence

In [3]:
endpoint_url = "https://query.wikidata.org/sparql"

sparql = SPARQLWrapper(endpoint_url)

sparql.setReturnFormat(JSON)

sparql.addCustomHttpHeader(
    "User-Agent",
    "HolyWellsResearch/1.0 (academic research; daria.stefan57@gmail.com)",
)

### Generalized query

In [4]:
# Wikidata namespace prefixes used to construct the SPARQL query.
WD = "http://www.wikidata.org/entity/"
P = "http://www.wikidata.org/prop/"
PS = "http://www.wikidata.org/prop/statement/"
PQ = "http://www.wikidata.org/prop/qualifier/"
PR = "http://www.wikidata.org/prop/reference/"
WDT = "http://www.wikidata.org/prop/direct/"

# qualifier properties (not reference)
qualifier_pids = ["P5817", "P5816"]


def build_statement_query(
    subject_qid: str,
    predicate_pid: str,
    *,
    qualifier_pids: Optional[Sequence[str]] = None,
    subject_instance_of_qid: Optional[str] = None,
    language: str = "en",
) -> str:

    prefixes = f"""
    PREFIX wd: <{WD}>
    PREFIX p: <{P}>
    PREFIX ps: <{PS}>
    PREFIX pq: <{PQ}>
    PREFIX pr: <{PR}>
    PREFIX wdt: <{WDT}>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX prov: <http://www.w3.org/ns/prov#>
    PREFIX schema: <http://schema.org/>
    """

    # Keep the original result fields and add description/property IDs.
    select_vars = """
        ?subject ?subjectLabel
        ?property ?propertyLabel
        ?object ?objectLabel
        ?objectInstanceOf ?objectInstanceOfLabel
        ?qualifierProperty ?qualifierPropertyLabel
        ?qualifierValue ?qualifierValueLabel
        ?qualifierDescription
        ?qualifierValueInstanceOf ?qualifierValueInstanceOfLabel
        ?statedIn ?statedInLabel
        ?statedInInstanceOf ?statedInInstanceOfLabel
        ?statedInProperty ?statedInObject ?statedInObjectLabel
    """

    where = f"""
        BIND(wd:{subject_qid} AS ?subject)
        BIND(wd:{predicate_pid} AS ?property)

        # Retrieve the selected main statement.
        ?subject p:{predicate_pid} ?statement .
        ?statement ps:{predicate_pid} ?object .

        OPTIONAL {{
            ?subject rdfs:label ?subjectLabel .
            FILTER(LANG(?subjectLabel) = "{language}")
        }}

        OPTIONAL {{
            ?property rdfs:label ?propertyLabel .
            FILTER(LANG(?propertyLabel) = "{language}")
        }}

        OPTIONAL {{
            ?object rdfs:label ?objectLabel .
            FILTER(LANG(?objectLabel) = "{language}")
        }}

        OPTIONAL {{
            ?object wdt:P31 ?objectInstanceOf .
            OPTIONAL {{
                ?objectInstanceOf rdfs:label ?objectInstanceOfLabel .
                FILTER(LANG(?objectInstanceOfLabel) = "{language}")
            }}
        }}

        # Retrieve qualifiers attached to this statement.
        OPTIONAL {{
            ?statement ?qualifierPredicate ?qualifierValue .

            FILTER(STRSTARTS(
                STR(?qualifierPredicate),
                "{PQ}"
            ))

            BIND(
                STRAFTER(STR(?qualifierPredicate), "/qualifier/")
                AS ?qualifierProperty
            )

            OPTIONAL {{
                ?qualifierPredicate rdfs:label ?qualifierPropertyLabel .
                FILTER(LANG(?qualifierPropertyLabel) = "{language}")
            }}

            OPTIONAL {{
                ?qualifierValue rdfs:label ?qualifierValueLabel .
                FILTER(LANG(?qualifierValueLabel) = "{language}")
                BIND(?qualifierValueLabel AS ?qualifierValueLabel)
            }}

            OPTIONAL {{
                ?qualifierValue schema:description ?description .
                FILTER(LANG(?description) = "{language}")
            }}

            OPTIONAL {{
                ?qualifierValue wdt:P31 ?qualifierValueInstanceOf .
                OPTIONAL {{
                    ?qualifierValueInstanceOf rdfs:label
                        ?qualifierValueInstanceOfLabel .
                    FILTER(LANG(?qualifierValueInstanceOfLabel) = "{language}")
                }}
            }}
        }}

        # Retrieve references and the source item's instance-of value.
        OPTIONAL {{
            ?statement prov:wasDerivedFrom ?reference .
            ?reference pr:P248 ?statedIn .

            OPTIONAL {{
                ?statedIn rdfs:label ?statedInLabel .
                FILTER(LANG(?statedInLabel) = "{language}")
            }}

            OPTIONAL {{
                ?statedIn wdt:P31 ?statedInInstanceOf .
                OPTIONAL {{
                    ?statedInInstanceOf rdfs:label
                        ?statedInInstanceOfLabel .
                    FILTER(LANG(?statedInInstanceOfLabel) = "{language}")
                }}
            }}

            # Retrieve direct properties of the cited source item.
            OPTIONAL {{
                ?statedIn ?sourcePredicate ?statedInObject .

                FILTER(
                    STRSTARTS(STR(?sourcePredicate), "{WDT}")
                    && ?sourcePredicate != wdt:P31
                )

                BIND(
                    STRAFTER(STR(?sourcePredicate), "/prop/direct/")
                    AS ?statedInProperty
                )

                OPTIONAL {{
                    ?statedInObject rdfs:label ?statedInObjectLabel .
                    FILTER(LANG(?statedInObjectLabel) = "{language}")
                }}
            }}
        }}
    """

    # Restrict the returned qualifier properties when requested - why would i still want this?
    if qualifier_pids is not None:
        if qualifier_pids:
            values = " ".join(
                f"pq:{pid}" for pid in qualifier_pids
            )
            where = where.replace(
                "?statement ?qualifierPredicate ?qualifierValue .",
                f"VALUES ?qualifierPredicate {{ {values} }}\n"
                "?statement ?qualifierPredicate ?qualifierValue .",
            )
        else:
            # Remove the entire optional qualifier block.
            start = where.index(
                "# Retrieve qualifiers attached to this statement."
            )
            end = where.index(
                "# Retrieve references and the source item's instance-of value."
            )
            where = where[:start] + where[end:]

    # Optionally restrict the subject by its P31 class - keep for final generalisation
    if subject_instance_of_qid:
        where += f"""
            ?subject wdt:P31 wd:{subject_instance_of_qid} .
        """

    return f"""
        {prefixes}
        SELECT DISTINCT {select_vars}
        WHERE {{
            {where}
        }}
    """

In [5]:
#Columbkille's Well - done
subject_qid = "Q126456441"

In [ ]:
#Bridget - done
subject_qid = "Q126454471"

In [ ]:
#Toberbride - done
subject_qid = "Q122258940"


In [ ]:
#Kenny - done
subject_qid = "Q114439798"


In [ ]:
# Lowrys well -done
subject_qid = "Q121840779"



In [ ]:
# Hermits well - done
subject_qid="Q126478185"

In [ ]:
# St Mogues well - done
subject_qid="Q126454603"

In [15]:
output_dir = Path("./xml_data/input")
output_dir.mkdir(parents=True, exist_ok=True)

# Retrieve all direct property IDs used by this subject.
properties_query = f"""
PREFIX wd: <{WD}>
PREFIX wikibase: <http://wikiba.se/ontology#>

SELECT DISTINCT ?property WHERE {{
    wd:{subject_qid} ?predicate ?value .
    ?property wikibase:directClaim ?predicate .
}}
ORDER BY ?property
"""

# request JSON for the property-discovery query.
sparql.setReturnFormat(JSON)
sparql.setQuery(properties_query)

properties_result = sparql.query().convert()

predicate_pids = sorted({
    row["property"]["value"].rsplit("/", 1)[-1]
    for row in properties_result["results"]["bindings"]
})

print(f"Found {len(predicate_pids)} properties for {subject_qid}")

# Run the general query for every property and save its XML.
skip_pids = {"P625"} # skip coordinates for now

for predicate_pid in predicate_pids:
    if predicate_pid in skip_pids:
        print(f"Skipped: {predicate_pid}")
        continue

    try:
        query = build_statement_query(
            subject_qid=subject_qid,
            predicate_pid=predicate_pid,
            language="en",
        )

        sparql.setQuery(query)
        sparql.setReturnFormat(XML)

        xml_result = sparql.query().convert()
        xml_bytes = xml_result.toxml(encoding="utf-8")

        output_path = output_dir / f"{subject_qid}_{predicate_pid}.xml"
        output_path.write_bytes(xml_bytes)

        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Failed for {predicate_pid}: {e}")


Found 21 properties for Q126456441
Saved: xml_data/input/Q126456441_P10.xml
Saved: xml_data/input/Q126456441_P11693.xml
Saved: xml_data/input/Q126456441_P131.xml
Saved: xml_data/input/Q126456441_P1325.xml
Saved: xml_data/input/Q126456441_P1343.xml
Saved: xml_data/input/Q126456441_P138.xml
Saved: xml_data/input/Q126456441_P1651.xml
Saved: xml_data/input/Q126456441_P17.xml
Saved: xml_data/input/Q126456441_P18.xml
Saved: xml_data/input/Q126456441_P195.xml
Saved: xml_data/input/Q126456441_P217.xml
Saved: xml_data/input/Q126456441_P2186.xml
Saved: xml_data/input/Q126456441_P276.xml
Saved: xml_data/input/Q126456441_P31.xml
Saved: xml_data/input/Q126456441_P373.xml
Saved: xml_data/input/Q126456441_P403.xml
Saved: xml_data/input/Q126456441_P4057.xml
Skipped: P625
Saved: xml_data/input/Q126456441_P708.xml
Saved: xml_data/input/Q126456441_P841.xml
Saved: xml_data/input/Q126456441_P973.xml


### Additional query for coordinates and scraping

## Run the x3ml mapper with the correct map for the input 

In [26]:
from pathlib import Path
import re
import subprocess

# Directories and configuration
input_dir = Path("xml_data/input")  # Directory containing the Q_P.xml files
mapping_dir = Path("xml_data/mappings")
output_dir = Path("xml_data/output")

engine_jar = Path(
    "X3ML/x3ml/target/x3ml-engine-2.2.3-SNAPSHOT-exejar.jar"
)
policy_file = Path("xml_data/xml_generator_policy/generator-policy.xml")

output_dir.mkdir(parents=True, exist_ok=True)

# Check required files and directories
for path in (engine_jar, policy_file):
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

if not input_dir.is_dir():
    raise FileNotFoundError(f"Input directory not found: {input_dir}")

if not mapping_dir.is_dir():
    raise FileNotFoundError(f"Mapping directory not found: {mapping_dir}")


# Index mappings by their exact P-code filename components.
# For example, P3342_P417.x3ml is indexed under P3342 and P417.
mappings_by_pid = {}

for mapping_file in mapping_dir.glob("*.x3ml"):
    for pid in mapping_file.stem.split("_"):
        if re.fullmatch(r"P\d+", pid):
            mappings_by_pid.setdefault(pid, []).append(mapping_file)


# Find all QID_PID.xml input files
xml_files = sorted(input_dir.glob("Q*_P*.xml"))

print(f"Found {len(xml_files)} XML files to process.\n")

succeeded = 0
failed = 0
skipped = 0

for xml_file in xml_files:
    # Expected filename format: Q126478185_P417.xml
    match = re.fullmatch(r"(Q\d+)_(P\d+)", xml_file.stem)

    if not match:
        print(f"Skipped (unexpected filename): {xml_file.name}")
        skipped += 1
        continue

    qid, pid = match.groups()

    # Find the mapping containing this exact property ID
    candidates = mappings_by_pid.get(pid, [])

    if not candidates:
        print(f"Skipped (no mapping found for {pid}): {xml_file.name}")
        skipped += 1
        continue

    if len(candidates) > 1:
        print(
            f"Skipped (multiple mappings contain {pid}): "
            f"{[p.name for p in candidates]}"
        )
        skipped += 1
        continue

    mapping_file = candidates[0]
    rdf_file = output_dir / f"{xml_file.stem}.rdf"

    command = [
        "java",
        "-jar", str(engine_jar.resolve()),
        "-i", str(xml_file.resolve()),
        "-x", str(mapping_file.resolve()),
        "-p", str(policy_file.resolve()),
        "-o", str(rdf_file.resolve()),
    ]

    print(f"Processing {xml_file.name}")
    print(f"  Property: {pid}")
    print(f"  Mapping:  {mapping_file.name}")

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=False,
        )

        if result.returncode == 0 and rdf_file.exists():
            print(f"  Saved:    {rdf_file}\n")
            succeeded += 1
        else:
            print(f"  FAILED (exit code {result.returncode})")
            if result.stdout.strip():
                print(result.stdout.strip())
            if result.stderr.strip():
                print(result.stderr.strip())
            print()
            failed += 1

    except OSError as e:
        print(f"  FAILED to execute Java: {e}\n")
        failed += 1


print("=" * 50)
print("Processing complete")
print(f"Successful: {succeeded}")
print(f"Failed:     {failed}")
print(f"Skipped:    {skipped}")
print(f"RDF output: {output_dir.resolve()}")

Found 20 XML files to process.

Processing Q126456441_P10.xml
  Property: P10
  Mapping:  P10_P18_P1810_P51.x3ml
  Saved:    xml_data/output/Q126456441_P10.rdf

Processing Q126456441_P11693.xml
  Property: P11693
  Mapping:  P2186_P4057_P11693_P10689_P14097_P4088.x3ml
  Saved:    xml_data/output/Q126456441_P11693.rdf

Processing Q126456441_P131.xml
  Property: P131
  Mapping:  P17_P131_P276_P708_P7959.x3ml
  Saved:    xml_data/output/Q126456441_P131.rdf

Processing Q126456441_P1325.xml
  Property: P1325
  Mapping:  P973_P1325_P6500.x3ml
  Saved:    xml_data/output/Q126456441_P1325.rdf

Processing Q126456441_P1343.xml
  Property: P1343
  Mapping:  P1343.x3ml
  Saved:    xml_data/output/Q126456441_P1343.rdf

Processing Q126456441_P138.xml
  Property: P138
  Mapping:  P138.x3ml
  Saved:    xml_data/output/Q126456441_P138.rdf

Processing Q126456441_P1651.xml
  Property: P1651
  Mapping:  P1651.x3ml
  Saved:    xml_data/output/Q126456441_P1651.rdf

Processing Q126456441_P17.xml
  Property: 

## join everything in a big rdf.

In [13]:
from pathlib import Path
import requests

from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import RDF, OWL

# --------------------------------------------------
# Configuration
# --------------------------------------------------

output_dir = Path("xml_data/output")
output_dir.mkdir(parents=True, exist_ok=True)

merged_rdf_file = output_dir / "merged_graph.rdf"
merged_ttl_file = output_dir / "merged_graph.ttl"

# Your project's ontology IRI
PROJECT_ONTOLOGY_IRI = URIRef(
    "http://ontology.holywells.link/ontology/"
)

# Official, versioned CIDOC CRM OWL ontology
CRM_OWL_URL = (
    "https://cidoc-crm.org/owl/7.1.3/"
    "CIDOC_CRM_v7.1.3.owl"
)

# True: include the actual CIDOC CRM ontology triples.
# False: only declare owl:imports.
INCLUDE_CRM_TRIPLES = False

# --------------------------------------------------
# 1. Parse and merge all individual RDF/XML files
# --------------------------------------------------

merged_graph = Graph()

crm_files = sorted(
    file
    for file in output_dir.glob("*.rdf")
    if file.name != merged_rdf_file.name
)

if not crm_files:
    raise FileNotFoundError(
        f"No RDF files found in {output_dir.resolve()}"
    )

print(f"Found {len(crm_files)} RDF files.")

parse_errors = []

for rdf_file in crm_files:
    try:
        merged_graph.parse(
            rdf_file.as_posix(),
            format="xml"
        )
        print(f"Merged: {rdf_file.name}")

    except Exception as exc:
        parse_errors.append((rdf_file.name, str(exc)))
        print(f"ERROR parsing {rdf_file.name}: {exc}")

if parse_errors:
    raise RuntimeError(
        f"{len(parse_errors)} RDF files failed to parse. "
        "Fix these errors before creating the final graph."
    )

data_triples = len(merged_graph)

print(f"\nTriples from your RDF files: {data_triples}")

# --------------------------------------------------
# 2. Declare the project ontology and its import
# --------------------------------------------------

merged_graph.add((
    PROJECT_ONTOLOGY_IRI,
    RDF.type,
    OWL.Ontology
))

merged_graph.add((
    PROJECT_ONTOLOGY_IRI,
    OWL.imports,
    URIRef(CRM_OWL_URL)
))

print(f"Declared ontology: {PROJECT_ONTOLOGY_IRI}")
print(f"CIDOC CRM import:  {CRM_OWL_URL}")

# --------------------------------------------------
# 3. Download and merge CIDOC CRM ontology triples
# --------------------------------------------------

crm_triple_count = 0

if INCLUDE_CRM_TRIPLES:
    print("\nDownloading CIDOC CRM ontology...")

    response = requests.get(CRM_OWL_URL, timeout=120)
    response.raise_for_status()

    crm_graph = Graph()
    crm_graph.parse(
        data=response.content,
        format="xml",
        publicID=CRM_OWL_URL
    )

    crm_triple_count = len(crm_graph)

    print(f"CIDOC CRM ontology triples: {crm_triple_count}")

    # Union of the project graph and the CIDOC CRM graph
    merged_graph += crm_graph

# --------------------------------------------------
# 4. Bind readable namespace prefixes
# --------------------------------------------------

merged_graph.bind(
    "crm",
    Namespace("http://www.cidoc-crm.org/cidoc-crm/")
)
merged_graph.bind(
    "hw",
    Namespace("http://ontology.holywells.link/ontology/")
)
merged_graph.bind("owl", OWL)

# --------------------------------------------------
# 5. Save the combined graph in two serializations
# --------------------------------------------------

merged_graph.serialize(
    destination=str(merged_rdf_file),
    format="xml",
    encoding="utf-8"
)

merged_graph.serialize(
    destination=str(merged_ttl_file),
    format="turtle",
    encoding="utf-8"
)

print("\nFinished.")
print(f"Original data triples: {data_triples}")
print(f"CIDOC CRM triples:     {crm_triple_count}")
print(f"Combined graph triples: {len(merged_graph)}")
print(f"RDF/XML: {merged_rdf_file.resolve()}")
print(f"Turtle:  {merged_ttl_file.resolve()}")

FileNotFoundError: No RDF files found in /home/daria/Uni/CAA/nfdi/Wikidata and Cidoc/Wikidata Gathering/holy_wells/xml_data/output

# wrapped for all

In [ ]:

from pathlib import Path
import re
import subprocess
import requests

from SPARQLWrapper import SPARQLWrapper, JSON, XML
from rdflib import Graph, Namespace, URIRef
from rdflib.namespace import RDF, OWL


# ============================================================
# 1. CONFIGURATION
# ============================================================

WD = "http://www.wikidata.org/entity/"

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"


# Find every item whose instance of (P31) is this class.
INSTANCE_OF_QID = "Q126443332"

BASE_DIR = Path(".")
INPUT_DIR = BASE_DIR / "xml_data" / "input"
MAPPING_DIR = BASE_DIR / "xml_data" / "mappings"
OUTPUT_DIR = BASE_DIR / "xml_data" / "output"

ENGINE_JAR = (
    BASE_DIR
    / "X3ML"
    / "x3ml"
    / "target"
    / "x3ml-engine-2.2.3-SNAPSHOT-exejar.jar"
)

POLICY_FILE = (
    BASE_DIR
    / "xml_data"
    / "xml_generator_policy"
    / "generator-policy.xml"
)

PROJECT_ONTOLOGY_IRI = URIRef(
    "http://ontology.holywells.link/ontology/"
)

CRM_OWL_URL = (
    "https://cidoc-crm.org/owl/7.1.3/"
    "CIDOC_CRM_v7.1.3.owl"
)

SKIP_PIDS = {"P625"}

# False: declare owl:imports without copying CRM triples.
# True: download and include the actual CRM ontology triples.
INCLUDE_CRM_TRIPLES = False

# Number of Wikidata items retrieved per discovery query.
QID_BATCH_SIZE = 500

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep one SPARQL client for the entire run.
sparql = SPARQLWrapper(WIKIDATA_ENDPOINT)
sparql.setTimeout(90)
sparql.addCustomHttpHeader(
    "User-Agent",
    "HolyWellsResearch/1.0 (academic research; daria.stefan57@gmail.com)",
)


# ============================================================
# 2. DISCOVER ALL INSTANCE QIDS
# ============================================================

def get_instance_qids(class_qid, batch_size=500):
    """
    Return all Wikidata QIDs whose direct P31 value is class_qid.

    Uses LIMIT/OFFSET pagination to retrieve results in batches.
    """
    if not re.fullmatch(r"Q\d+", class_qid):
        raise ValueError(f"Invalid class QID: {class_qid}")

    qids = set()
    offset = 0

    while True:
        query = f"""
        SELECT DISTINCT ?item WHERE {{
            ?item wdt:P31 wd:{class_qid} .
            FILTER(STRSTARTS(STR(?item),
                "http://www.wikidata.org/entity/Q"))
        }}
        ORDER BY ?item
        LIMIT {int(batch_size)}
        OFFSET {offset}
        """

        sparql.setQuery(query)
        sparql.setReturnFormat(JSON)

        results = sparql.query().convert()["results"]["bindings"]

        if not results:
            break

        for row in results:
            qid = row["item"]["value"].rsplit("/", 1)[-1]

            if re.fullmatch(r"Q\d+", qid):
                qids.add(qid)

        print(
            f"Discovered {len(qids)} items "
            f"(current offset: {offset})"
        )

        if len(results) < batch_size:
            break

        offset += batch_size

    return sorted(qids)


# ============================================================
# 3. DISCOVER DIRECT PROPERTIES FOR ONE ITEM
# ============================================================

def get_property_ids(subject_qid):
    """
    Discover direct Wikidata property IDs used by a subject.
    """
    query = f"""
    PREFIX wd: <{WD}>
    PREFIX wikibase: <http://wikiba.se/ontology#>

    SELECT DISTINCT ?property WHERE {{
        wd:{subject_qid} ?predicate ?value .
        ?property wikibase:directClaim ?predicate .
    }}
    ORDER BY ?property
    """

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    results = sparql.query().convert()

    property_ids = {
        row["property"]["value"].rsplit("/", 1)[-1]
        for row in results["results"]["bindings"]
    }

    return sorted(
        pid for pid in property_ids
        if re.fullmatch(r"P\d+", pid)
    )


# ============================================================
# 4. RETRIEVE AND SAVE XML FOR ONE ITEM
# ============================================================

def fetch_subject_xml(subject_qid):
    """
    Retrieve all supported property XML files for one QID.

    Uses the existing build_statement_query function.
    P625 coordinates are skipped.
    """
    property_ids = get_property_ids(subject_qid)

    print(
        f"\n{subject_qid}: "
        f"found {len(property_ids)} properties"
    )

    saved = 0
    skipped = 0
    failed = 0

    for pid in property_ids:
        if pid in SKIP_PIDS:
            print(f"  Skipped property: {pid}")
            skipped += 1
            continue

        output_path = INPUT_DIR / f"{subject_qid}_{pid}.xml"

        try:
            query = build_statement_query(
                subject_qid=subject_qid,
                predicate_pid=pid,
                language="en",
            )

            sparql.setQuery(query)
            sparql.setReturnFormat(XML)

            xml_result = sparql.query().convert()
            xml_bytes = xml_result.toxml(encoding="utf-8")

            output_path.write_bytes(xml_bytes)

            print(f"  Saved XML: {output_path.name}")
            saved += 1

        except Exception as exc:
            print(f"  XML FAILED for {pid}: {exc}")
            failed += 1

    return {
        "properties": len(property_ids),
        "xml_saved": saved,
        "skipped": skipped,
        "failed": failed,
    }


# ============================================================
# 5. INDEX X3ML MAPPINGS
# ============================================================

def index_mappings(mapping_dir):
    """
    Index mapping files by every exact P-code in their filename.

    Example:
        P3342_P417.x3ml -> P3342 and P417
    """
    mappings_by_pid = {}

    for mapping_file in mapping_dir.glob("*.x3ml"):
        for pid in mapping_file.stem.split("_"):
            if re.fullmatch(r"P\d+", pid):
                mappings_by_pid.setdefault(pid, []).append(
                    mapping_file.resolve()
                )

    return mappings_by_pid


# ============================================================
# 6. GENERATE RDF FOR ONE ITEM
# ============================================================

def generate_subject_rdf(
    subject_qid,
    mappings_by_pid,
    engine_jar,
    policy_file,
):
    """
    Process this subject's XML files using X3ML.

    RDF files are isolated in output/<QID>/.
    """
    subject_output_dir = OUTPUT_DIR / subject_qid
    subject_output_dir.mkdir(parents=True, exist_ok=True)

    xml_files = sorted(
        INPUT_DIR.glob(f"{subject_qid}_P*.xml")
    )

    succeeded = 0
    failed = 0
    skipped = 0

    print(f"\n{subject_qid}: {len(xml_files)} XML files to process")

    for xml_file in xml_files:
        match = re.fullmatch(
            rf"({re.escape(subject_qid)})_(P\d+)",
            xml_file.stem,
        )

        if not match:
            print(f"  Skipped unexpected filename: {xml_file.name}")
            skipped += 1
            continue

        _, pid = match.groups()

        if pid in SKIP_PIDS:
            skipped += 1
            continue

        candidates = mappings_by_pid.get(pid, [])

        if len(candidates) == 0:
            print(f"  Skipped: no mapping for {pid}")
            skipped += 1
            continue

        if len(candidates) > 1:
            print(
                f"  Skipped: multiple mappings for {pid}: "
                f"{[p.name for p in candidates]}"
            )
            skipped += 1
            continue

        mapping_file = candidates[0]
        rdf_file = subject_output_dir / f"{xml_file.stem}.rdf"

        command = [
            "java",
            "-jar", str(engine_jar),
            "-i", str(xml_file.resolve()),
            "-x", str(mapping_file),
            "-p", str(policy_file),
            "-o", str(rdf_file.resolve()),
        ]

        print(f"  Processing {xml_file.name} with {mapping_file.name}")

        try:
            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                check=False,
            )

            if result.returncode == 0 and rdf_file.is_file():
                print(f"    Saved: {rdf_file.name}")
                succeeded += 1
            else:
                print(
                    f"    FAILED (exit code {result.returncode})"
                )

                if result.stdout.strip():
                    print(result.stdout.strip())

                if result.stderr.strip():
                    print(result.stderr.strip())

                failed += 1

        except OSError as exc:
            print(f"    Java execution failed: {exc}")
            failed += 1

    return {
        "xml_files": len(xml_files),
        "rdf_saved": succeeded,
        "failed": failed,
        "skipped": skipped,
        "output_dir": subject_output_dir,
    }


# ============================================================
# 7. MERGE ONE ITEM'S RDF FILES
# ============================================================

def merge_subject_rdf(
    subject_qid,
    subject_output_dir,
    include_crm_triples=False,
):
    """
    Merge only the RDF files belonging to this subject.

    Creates:
        output/<QID>/merged_graph.rdf
        output/<QID>/merged_graph.ttl
    """
    merged_rdf_file = subject_output_dir / "merged_graph.rdf"
    merged_ttl_file = subject_output_dir / "merged_graph.ttl"

    rdf_files = sorted(
        file
        for file in subject_output_dir.glob("*.rdf")
        if file.name != merged_rdf_file.name
    )

    if not rdf_files:
        print(f"  No RDF files to merge for {subject_qid}")
        return {
            "success": False,
            "triples": 0,
            "rdf_file": None,
            "ttl_file": None,
        }

    merged_graph = Graph()
    parse_errors = []

    for rdf_file in rdf_files:
        try:
            merged_graph.parse(
                rdf_file.as_posix(),
                format="xml",
            )
        except Exception as exc:
            parse_errors.append((rdf_file.name, str(exc)))
            print(f"  Parse error in {rdf_file.name}: {exc}")

    if parse_errors:
        print(
            f"  Merge aborted: {len(parse_errors)} RDF files "
            "could not be parsed."
        )

        return {
            "success": False,
            "triples": len(merged_graph),
            "rdf_file": None,
            "ttl_file": None,
        }

    data_triples = len(merged_graph)

    # Declare the project ontology and its CRM dependency.
    merged_graph.add((
        PROJECT_ONTOLOGY_IRI,
        RDF.type,
        OWL.Ontology,
    ))

    merged_graph.add((
        PROJECT_ONTOLOGY_IRI,
        OWL.imports,
        URIRef(CRM_OWL_URL),
    ))

    crm_triples = 0

    if include_crm_triples:
        print("  Downloading CIDOC CRM ontology...")

        response = requests.get(CRM_OWL_URL, timeout=120)
        response.raise_for_status()

        crm_graph = Graph()
        crm_graph.parse(
            data=response.content,
            format="xml",
            publicID=CRM_OWL_URL,
        )

        crm_triples = len(crm_graph)
        merged_graph += crm_graph

    # Bind namespace prefixes.
    merged_graph.bind(
        "crm",
        Namespace("http://www.cidoc-crm.org/cidoc-crm/"),
    )
    merged_graph.bind(
        "hw",
        Namespace("http://ontology.holywells.link/ontology/"),
    )
    merged_graph.bind("owl", OWL)

    # Serialize the merged graph.
    merged_graph.serialize(
        destination=str(merged_rdf_file),
        format="xml",
        encoding="utf-8",
    )

    merged_graph.serialize(
        destination=str(merged_ttl_file),
        format="turtle",
        encoding="utf-8",
    )

    print(f"  Data triples: {data_triples}")
    print(f"  CRM triples added: {crm_triples}")
    print(f"  Total triples: {len(merged_graph)}")
    print(f"  RDF/XML: {merged_rdf_file}")
    print(f"  Turtle:  {merged_ttl_file}")

    return {
        "success": True,
        "triples": len(merged_graph),
        "rdf_file": merged_rdf_file,
        "ttl_file": merged_ttl_file,
    }


# ============================================================
# 8. PROCESS ONE ITEM THROUGH THE COMPLETE PIPELINE
# ============================================================

def process_subject(subject_qid, mappings_by_pid):
    """
    Run XML retrieval, X3ML conversion and RDF merging for one QID.
    """
    print("\n" + "=" * 70)
    print(f"PROCESSING {subject_qid}")
    print("=" * 70)

    report = {
        "qid": subject_qid,
        "xml_saved": 0,
        "rdf_saved": 0,
        "failed": 0,
        "skipped": 0,
        "merged": False,
        "triples": 0,
        "error": None,
    }

    try:
        # Stage 1: Wikidata -> property XML.
        xml_stats = fetch_subject_xml(subject_qid)

        report["xml_saved"] = xml_stats["xml_saved"]
        report["failed"] += xml_stats["failed"]
        report["skipped"] += xml_stats["skipped"]

        # Stage 2: property XML -> individual RDF files.
        rdf_stats = generate_subject_rdf(
            subject_qid=subject_qid,
            mappings_by_pid=mappings_by_pid,
            engine_jar=ENGINE_JAR.resolve(),
            policy_file=POLICY_FILE.resolve(),
        )

        report["rdf_saved"] = rdf_stats["rdf_saved"]
        report["failed"] += rdf_stats["failed"]
        report["skipped"] += rdf_stats["skipped"]

        # Stage 3: merge RDF for this QID only.
        merge_stats = merge_subject_rdf(
            subject_qid=subject_qid,
            subject_output_dir=rdf_stats["output_dir"],
            include_crm_triples=INCLUDE_CRM_TRIPLES,
        )

        report["merged"] = merge_stats["success"]
        report["triples"] = merge_stats["triples"]

    except Exception as exc:
        report["error"] = str(exc)
        print(f"SUBJECT FAILED: {subject_qid}: {exc}")

    return report


# ============================================================
# 9. MAIN BATCH LOOP
# ============================================================

def main():
    """
    Discover every P31 instance and run the complete pipeline.
    """
    # Validate required project resources before processing.
    for path in (ENGINE_JAR, POLICY_FILE):
        if not path.is_file():
            raise FileNotFoundError(
                f"Required file not found: {path.resolve()}"
            )

    if not MAPPING_DIR.is_dir():
        raise FileNotFoundError(
            f"Mapping directory not found: {MAPPING_DIR.resolve()}"
        )

    if not callable(build_statement_query):
        raise RuntimeError(
            "Define build_statement_query() before running main()."
        )

    print(f"Discovering instances of {INSTANCE_OF_QID}...")

    # Stage 0: discover QIDs from Wikidata.
    qids = get_instance_qids(
        class_qid=INSTANCE_OF_QID,
        batch_size=QID_BATCH_SIZE,
    )

    if not qids:
        print("No instances found. Nothing to process.")
        return []

    print(f"\nTotal discovered QIDs: {len(qids)}")

    # Build mapping index once, not once per item.
    mappings_by_pid = index_mappings(MAPPING_DIR)

    print(f"Indexed {len(mappings_by_pid)} property IDs in mappings.")

    reports = []

    # Run the entire pipeline for every discovered item.
    for index, qid in enumerate(qids, start=1):
        print(f"\nITEM {index}/{len(qids)}")

        report = process_subject(
            subject_qid=qid,
            mappings_by_pid=mappings_by_pid,
        )

        reports.append(report)

    # Final batch report.
    print("\n" + "=" * 70)
    print("BATCH PROCESSING COMPLETE")
    print("=" * 70)

    merged_count = sum(r["merged"] for r in reports)
    failed_count = sum(r["failed"] for r in reports)
    skipped_count = sum(r["skipped"] for r in reports)
    xml_count = sum(r["xml_saved"] for r in reports)
    rdf_count = sum(r["rdf_saved"] for r in reports)

    print(f"Items discovered: {len(qids)}")
    print(f"Items merged:     {merged_count}")
    print(f"XML files saved:  {xml_count}")
    print(f"RDF files saved:  {rdf_count}")
    print(f"Failures:         {failed_count}")
    print(f"Skipped files:    {skipped_count}")

    print("\nPer-item results:")

    for r in reports:
        status = "OK" if r["merged"] else "INCOMPLETE"

        print(
            f"{r['qid']}: {status}, "
            f"XML={r['xml_saved']}, "
            f"RDF={r['rdf_saved']}, "
            f"triples={r['triples']}"
        )

        if r["error"]:
            print(f"  Error: {r['error']}")

    print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")

    return reports


# ============================================================
# 10. EXECUTE
# ============================================================

if __name__ == "__main__":
    reports = main()


Discovering instances of Q126443332...
Discovered 242 items (current offset: 0)

Total discovered QIDs: 242
Indexed 35 property IDs in mappings.

ITEM 1/242

PROCESSING Q114439798

Q114439798: found 17 properties
  Saved XML: Q114439798_P10.xml
  Saved XML: Q114439798_P110.xml
  Saved XML: Q114439798_P11693.xml
  Saved XML: Q114439798_P131.xml
  Saved XML: Q114439798_P1325.xml
  Saved XML: Q114439798_P1343.xml
  Saved XML: Q114439798_P138.xml
  Saved XML: Q114439798_P17.xml
  Saved XML: Q114439798_P18.xml
  Saved XML: Q114439798_P195.xml
  Saved XML: Q114439798_P217.xml
  Saved XML: Q114439798_P31.xml
  Saved XML: Q114439798_P373.xml
  Saved XML: Q114439798_P4057.xml
  Skipped property: P625
  Saved XML: Q114439798_P708.xml
  Saved XML: Q114439798_P973.xml

Q114439798: 16 XML files to process
  Processing Q114439798_P10.xml with P10_P18_P1810_P51.x3ml
    Saved: Q114439798_P10.rdf
  Processing Q114439798_P110.xml with P110.x3ml
    Saved: Q114439798_P110.rdf
  Processing Q114439798_P11

In [ ]:

from pathlib import Path

from rdflib import Graph, URIRef, Namespace
from rdflib.namespace import RDF, OWL


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("xml_data/output")

BIG_GRAPH_RDF = OUTPUT_DIR / "all_merged_graphs.rdf"
BIG_GRAPH_TTL = OUTPUT_DIR / "all_merged_graphs.ttl"

PROJECT_ONTOLOGY_IRI = URIRef(
    "http://ontology.holywells.link/ontology"
)

# Ontologies to import.
# These IRIs are recorded as owl:imports only.
# Their triples are NOT downloaded or added to the graph.
ONTOLOGY_IMPORTS = [
    "http://www.cidoc-crm.org/cidoc-crm/",
    "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "http://www.w3.org/2004/02/skos/core",
    "http://www.w3.org/2006/time#2016",
]


# ============================================================
# 1. FIND ALL PER-ITEM MERGED GRAPHS
# ============================================================

merged_files = sorted(
    file
    for file in OUTPUT_DIR.rglob("merged_graph.rdf")
    if file.resolve() != BIG_GRAPH_RDF.resolve()
)

if not merged_files:
    raise FileNotFoundError(
        f"No per-item merged_graph.rdf files found under "
        f"{OUTPUT_DIR.resolve()}"
    )

print(f"Found {len(merged_files)} per-item merged graphs.")


# ============================================================
# 2. PARSE AND MERGE ALL GRAPHS
# ============================================================

big_graph = Graph()
parse_errors = []

for rdf_file in merged_files:
    try:
        big_graph.parse(
            rdf_file.as_posix(),
            format="xml",
        )
        print(f"Merged: {rdf_file}")

    except Exception as exc:
        parse_errors.append((rdf_file, str(exc)))
        print(f"ERROR parsing {rdf_file}: {exc}")


# Do not silently create an incomplete final graph.
if parse_errors:
    details = "\n".join(
        f"  {path}: {error}"
        for path, error in parse_errors
    )
    raise RuntimeError(
        f"{len(parse_errors)} merged graph(s) failed to parse:\n"
        f"{details}"
    )

data_triples = len(big_graph)

print(f"\nTriples merged from per-item graphs: {data_triples}")


# ============================================================
# 3. DECLARE THE PROJECT ONTOLOGY AND ITS IMPORTS
# ============================================================

big_graph.add((
    PROJECT_ONTOLOGY_IRI,
    RDF.type,
    OWL.Ontology,
))

# Match the ontology metadata in your supplied RDF/XML.
big_graph.add((
    PROJECT_ONTOLOGY_IRI,
    OWL.versionIRI,
    URIRef("http://ontology.holywells.link/ontology/"),
))

for ontology_iri in ONTOLOGY_IMPORTS:
    big_graph.add((
        PROJECT_ONTOLOGY_IRI,
        OWL.imports,
        URIRef(ontology_iri),
    ))

print(f"Declared ontology: {PROJECT_ONTOLOGY_IRI}")
print("Declared owl:imports:")
for ontology_iri in ONTOLOGY_IMPORTS:
    print(f"  {ontology_iri}")


# ============================================================
# 4. BIND NAMESPACE PREFIXES
# ============================================================

big_graph.bind(
    "hw",
    Namespace("http://ontology.holywells.link/ontology/")
)
big_graph.bind(
    "crm",
    Namespace("http://www.cidoc-crm.org/cidoc-crm/")
)
big_graph.bind(
    "CRMgeo",
    Namespace("http://www.ics.forth.gr/isl/CRMgeo/")
)
big_graph.bind(
    "wdQ",
    Namespace("http://wikidata.org/entity/")
)
big_graph.bind(
    "skos",
    Namespace("http://www.w3.org/2004/02/skos/core#")
)
big_graph.bind("owl", OWL)


# ============================================================
# 5. SAVE THE FINAL GRAPH IN BOTH FORMATS
# ============================================================

big_graph.serialize(
    destination=str(BIG_GRAPH_RDF),
    format="xml",
    encoding="utf-8",
)

big_graph.serialize(
    destination=str(BIG_GRAPH_TTL),
    format="turtle",
    encoding="utf-8",
)


# ============================================================
# 6. REPORT
# ============================================================

print("\n" + "=" * 60)
print("FINAL GRAPH CREATED")
print("=" * 60)
print(f"Input graphs: {len(merged_files)}")
print(f"Data triples before ontology declarations: {data_triples}")
print(f"Final triples, including ontology declarations: {len(big_graph)}")
print(f"Ontology imports declared: {len(ONTOLOGY_IMPORTS)}")
print(f"RDF/XML: {BIG_GRAPH_RDF.resolve()}")
print(f"Turtle:  {BIG_GRAPH_TTL.resolve()}")
print("Imported ontology triples were not downloaded or added.")
